# Export FLOAT modelu Model61.1 do ONNX (ARBot3)

Vezme z Google Drive uložený model `Model61.1_0.9545764327049255.h5` a vyrobí z něj
`.onnx`, který jde pustit v ARBot3. Cíl je jediné číslo: **kolik ten model umí ve floatu**,
měřeno **týmž měřidlem** jako dnešní int8 varianta
(`ARBot.Analyze backproject --truth=models/testset`).

**Proč to potřebujeme.** Int8 model z `models/` dává na 50snímkové testovací sadě přesnost
**88,2 %**, ale číslo v názvu těch vah slibuje **95,5 %** (a je to opravdu číslo z testovací
sady — `Trainer` do jména ukládá `val_sparse_categorical_accuracy`). Ten rozdíl 7,3 p. b.
může být kvantizace, nebo taky jen to, že `.tflite` v repu vznikl z **jiného checkpointu**
(`SaveTFLite` exportuje ten model, který je právě načtený). Tenhle notebook to rozhodne,
protože bere **přesně ten checkpoint, ke kterému se to číslo váže**.

## Co jsou ty „dynamické tvary" a proč kvůli nim nejde použít `.tflite`

Dekodér modelu **pětkrát zdvojnásobí rozlišení** (`UpSampling2D(size=(2,2))`).
Když se model převáděl do TFLite, konvertor se volal takto:

```python
tf.lite.TFLiteConverter.from_keras_model(model)   # bez udání vstupního tvaru
```

Protože mu **nikdo neřekl, jak velký vstup přijde**, nemohl si spočítat, jak velké budou
obrázky uvnitř. Místo instrukce *„zvětši na 8×8"* proto do souboru zapsal instrukci
*„podívej se, jak velký je vstup, vynásob dvěma a zvětši na to"* — tedy řetěz operací
`Shape → StridedSlice → Mul → ResizeNearestNeighbor`, a to pětkrát.

**Přirovnání:** je to jako recept, který místo „peč ve formě 24 cm" říká „peč ve formě
dvakrát širší, než byla ta předchozí". Upéct se podle toho dá, ale **z receptu se nedá
přečíst, jak velký bude výsledek** — to se pozná teprve za běhu.

Dva důsledky:

1. **`tf2onnx` to nepřevede**, protože potřebuje tvary znát dopředu.
2. Soubor sám **deklaruje výstup `[1,1,1,2]`**, tedy obrázek 1×1 pixel. To je nesmysl —
   je to jen výplň, protože skutečnou velikost konvertor neznal.

**Léčba je jedna věta:** při exportu se konvertoru **řekne přesný vstupní tvar**
(`tf.TensorSpec((1, 128, 128, 3))`). Pak si všechny vnitřní velikosti spočítá sám a zapíše je
jako konstanty („zvětši na 8×8"), a ten řetěz `Shape → … → Mul` z grafu zmizí jako mrtvý kód.
Proto se tady **nekonvertuje `.tflite`, ale exportuje přímo z Kerasu** — jinému postupu se
ty dynamické tvary odstranit nedají, protože v tom `.tflite` už jsou zapečené.

⚠️ **Mimochodem: ani `Model61.1.tflite` není „float".** Ta větev `SaveTFLite` používá
`optimizations=[tf.lite.Optimize.DEFAULT]` **bez** `representative_dataset`, což je
dynamic-range kvantizace — váhy jsou int8, počítá se ve floatu. Referenční float model tedy
v repu není žádný a tenhle notebook je jediná cesta, jak ho získat.

## Jak použít

1. *Runtime → Run all* a povolit připojení Google Drive.
2. Stáhne se `Model61.1_float.onnx` — ulož ho do `models/` v repu.
3. Změř tímtéž měřidlem jako int8:
   `ARBot.Analyze backproject --truth=models/testset --model=models/Model61.1_float.onnx`

⚠️ **Kde to může zaškobrtnout.** Verze balíčků se tu **záměrně nepřipínají** — pinning
opsaný z prostředí pro `models/tflite2onnx.py` (WSL, TF 2.15) skončí v dnešním Colabu na
`ResolutionImpossible`. Na výsledku verze nic nemění: tvary fixuje `TensorSpec`. Druhá věc je
**legacy Keras** — `TF_USE_LEGACY_KERAS` musí být nastavené **dřív, než se poprvé naimportuje TensorFlow**; po instalaci balíčků proto někdy nezbývá než *Runtime → Restart session* a spustit znovu. První cela to sama zkontroluje a napíše.

In [ ]:
# ⚠️ VERZE SE ZAMERNE NEPRIPINAJI. Prvni verze tady mela tf2onnx==1.16.1 a onnx==1.16.1
# opsane z prostredi pro models/tflite2onnx.py (WSL, TF 2.15) - a v dnesnim Colabu to konci
# na "ResolutionImpossible", protoze onnxruntime 1.27+ chce novejsi onnx. Na vysledku ty
# verze nemeni nic: tvary fixuje TensorSpec pri exportu, ne verze konvertoru.
!pip install -q tf_keras tf2onnx onnx onnxruntime

# Keras 3 (dnesni Colab) stare .h5 casto nenacte, proto se zapina LEGACY Keras - a musi se
# to udelat PRED importem tensorflow, jinak uz je pozde.
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import glob
import numpy as np
import tensorflow as tf

print('TensorFlow', tf.__version__)
# tf.keras.__version__ pod legacy Kerasem NEEXISTUJE (modul tf_keras.api._v2.keras ho nema),
# takze se cte z tf_keras. Zaroven se overi, ze prepinac zabral - kdyz ne, .h5 se nenacte
# a je lepsi to vedet tady nez o dve cely dal.
try:
  import tf_keras
  print('legacy keras:', tf_keras.__version__)
except Exception as e:
  print('!! tf_keras neni k dispozici (%s: %s)' % (type(e).__name__, e))

print('tf.keras je:', tf.keras.__name__)
if 'tf_keras' not in tf.keras.__name__:
  print('!! POZOR: legacy Keras se NEZAPNUL. Restartuj session (Runtime -> Restart session)')
  print('   a pust cely notebook znovu - promenna TF_USE_LEGACY_KERAS musi byt nastavena')
  print('   driv, nez se tensorflow poprve naimportuje.')

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

# Soubor se hleda, ne hada - cesta ke slozce se v Drive mohla zmenit.
VZOR = '/content/gdrive/MyDrive/**/Model61.1*0.9545764327049255*.h5'
nalezene = glob.glob(VZOR, recursive=True)
print('nalezeno %d souboru vzorem %s' % (len(nalezene), VZOR))
for f in nalezene:
  print('  %s  (%.1f MB)' % (f, os.path.getsize(f) / 1024 / 1024))

if not nalezene:
  # Zaloha: vypis, co v models/ vubec je, at je videt cim to nahradit.
  print()
  print('NENALEZENO. Co je na Drive v models/:')
  for f in sorted(glob.glob('/content/gdrive/MyDrive/models/**/*.h5', recursive=True))[:40]:
    print('  ', f)
  raise RuntimeError('Uprav VZOR na skutecnou cestu k .h5 souboru.')

H5 = nalezene[0]

In [ ]:
# .h5 ulozeny pres m.save() nese ARCHITEKTURU I VAHY, takze se nemusi znovu stavet
# GenericModel20 z velkeho notebooku. Kdyby load_model presto selhal (Keras 3 umi u starych
# souboru prekvapit), je zaloha popsana v hlasce niz.
try:
  model = tf.keras.models.load_model(H5, compile=False)
except Exception as e:
  raise RuntimeError(
      'load_model selhal (%s: %s).\n\n'
      'ZALOHA: vloz tuhle a nasledujici celu do SemanticSegmentation.ipynb POD cely, ktere\n'
      'definuji GenericModel20, a model postav znovu:\n'
      '  model = GenericModel20("Model61.1", (128, 128, 3), 2, 1, 0.99, 0.2, "float32",\n'
      '      [[8,16],[16,32],[32,64],[64,128],[128,128],[128,256]],\n'
      '      [[256,128],[128,128],[128,64],[64,32],[32,16],[16,16]],\n'
      '      [False,False,True,True,True,True], [])\n'
      '  model.load_weights(H5)\n' % (type(e).__name__, e))

print('nacteno:', model.name)
print('vstup :', model.input_shape)
print('vystup:', model.output_shape)
print('vahovych tenzoru:', len(model.weights))

# Kontrola, ze je to opravdu ten model, ktery cekame: 128x128x3 -> 128x128x2.
if tuple(model.input_shape[1:]) != (128, 128, 3) or tuple(model.output_shape[1:]) != (128, 128, 2):
  raise RuntimeError('Necekany tvar modelu - dal by se export, ktery ARBot3 neprijme.')

In [ ]:
import tf2onnx

ONNX = 'Model61.1_float.onnx'

# TOHLE JE TA PODSTATNA VEC: pevny vstupni tvar. Diky nemu si exporter spocita vsechny
# vnitrni velikosti sam a zapise je jako konstanty, takze v grafu NEVZNIKNE retez
# Shape -> StridedSlice -> Mul -> Resize, kterym se to pokazilo pri prevodu do TFLite.
spec = (tf.TensorSpec((1, 128, 128, 3), tf.float32, name='input'),)

try:
  tf2onnx.convert.from_keras(model, input_signature=spec, opset=13, output_path=ONNX)
except Exception as e:
  # Zaloha pres SavedModel - tatataz vec, jen jinou cestou; concrete_function ma tvar
  # zafixovany taky, takze vysledek ma byt stejny.
  print('from_keras selhal (%s: %s), zkousim pres SavedModel' % (type(e).__name__, e))
  fce = tf.function(lambda x: model(x)).get_concrete_function(spec[0])
  tf.saved_model.save(model, 'sm', signatures={'serving_default': fce})
  os.system('python -m tf2onnx.convert --saved-model sm --output %s --opset 13' % ONNX)

print('velikost: %.2f MB' % (os.path.getsize(ONNX) / 1024 / 1024))

In [ ]:
# OVERENI, ze dynamicke tvary opravdu zmizely - netvrdit to, zmerit to.
import onnx
import onnxruntime as ort

m = onnx.load(ONNX)
dynamicke = [n.name for n in m.graph.node if n.op_type in ('Shape', 'StridedSlice', 'Slice')]
print('uzlu celkem:', len(m.graph.node))
print('uzlu typu Shape/Slice (zbytky pocitani tvaru za behu):', len(dynamicke))
if dynamicke:
  print('  ', dynamicke[:10])

sess = ort.InferenceSession(ONNX, providers=['CPUExecutionProvider'])
vst, vys = sess.get_inputs()[0], sess.get_outputs()[0]
print('vstup :', vst.name, vst.shape, vst.type)
print('vystup:', vys.name, vys.shape, vys.type)

staticke = all(isinstance(d, int) for d in vys.shape)
print()
print('vsechny rozmery vystupu jsou cisla:', staticke, '  <- tohle musi byt True')
if not staticke:
  print('!! Tvary zustaly dynamicke - ARBot3 model NEPRIJME (OnnxBackProject cte rozmery')
  print('   z metadat). Dalsi krok by byl onnxsim / onnx.tools.update_model_dims.')

# Beh na sumu: dokazuje, ze model skutecne pocita a vraci 128x128x2 v rozsahu 0..1.
y = sess.run(None, {vst.name: np.random.rand(1, 128, 128, 3).astype(np.float32)})[0]
print()
print('vystup na sumu: tvar', y.shape, '| min %.3f max %.3f | soucet kanalu %.3f az %.3f'
      % (y.min(), y.max(), y.sum(axis=-1).min(), y.sum(axis=-1).max()))
print('(soucet kanalu neni 1 - model konci sigmoidou; ARBot3 si ho normalizuje sam)')

In [ ]:
from google.colab import files
files.download(ONNX)
print('Uloz do models/ v repu a zmer TYMZ meridlem jako int8:')
print('  ARBot.Analyze backproject --truth=models/testset --model=models/%s' % ONNX)